# 依赖和配置

In [ ]:
# 单元格 1：依赖和配置

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import time
import uuid
import warnings

import cloudpickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
import xgboost as xgb
import lightgbm as lgb

from scipy.stats import rankdata
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from threadpoolctl import threadpool_limits

PROJECT_DIR = Path(r"E:\document-prefer\python for finanace\multi-model-bactest")
DATA_ROOT = PROJECT_DIR / "data" / "processed"
OUTPUT_ROOT = DATA_ROOT / "model_training"

MODEL_NAMES = [
    "ols",
    "lasso",
    "random_forest",
    "xgboost",
    "lightgbm",
    "mlp",
]

RANDOM_STATE = 2026
N_JOBS = 4

# 六模型使用同一组训练样本；改为 None 使用全部训练样本。
TRAIN_MAX_ROWS = 100_000

RUN_TUNING = True
SELECTION_METRIC = "rank_ic"  # "rank_ic" 或 "rmse"
MIN_CROSS_SECTION = 30
PREDICT_BATCH_ROWS = 100_000

# 仅在线性模型和 MLP 的训练 Pipeline 内拟合。
CLIP_QUANTILES = (0.01, 0.99)

# 默认保留测试集；不会用于调参。
EVALUATE_TEST = False

PARAM_GRIDS = {
    "ols": [{}],
    "lasso": {
        "alpha": [1e-5, 5e-5, 1e-4],
    },
    "random_forest": {
        "max_depth": [8, 12],
    },
    "xgboost": {
        "max_depth": [3, 5],
    },
    "lightgbm": {
        "num_leaves": [15, 31],
    },
    "mlp": {
        "hidden_layer_sizes": [(32,), (64, 32)],
    },
}

assert SELECTION_METRIC in {"rank_ic", "rmse"}

VERSIONS = {
    "sklearn": sklearn.__version__,
    "xgboost": xgb.__version__,
    "lightgbm": lgb.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}

pd.Series(VERSIONS, name="version")

# 固定筛选版本、模型数据版本和原始因子版本

In [ ]:
# 单元格 2：固定筛选版本、模型数据版本和原始因子版本

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def latest_run(root):
    root = Path(root)
    path = (root / read_json(root / "latest_run.json")["run_dir"]).resolve()

    if path.parent != root.resolve():
        raise ValueError("latest_run.json 指向了根目录之外")

    return path


SELECTION_RUN = latest_run(DATA_ROOT / "factor_selection")
SELECTION_SOURCE = read_json(SELECTION_RUN / "source.json")
SELECTION_META = read_json(SELECTION_RUN / "metadata.json")
FEATURES = read_json(SELECTION_RUN / "selected_factors.json")

# 使用筛选时绑定的数据版本，而不是另外读取 model_dataset/latest。
MODEL_DATA_RUN = Path(SELECTION_SOURCE["model_run"])
MODEL_META = read_json(MODEL_DATA_RUN / "metadata.json")
MODEL_CONFIG = read_json(MODEL_DATA_RUN / "config.json")

ALPHA_RUN = Path(MODEL_META["source_run"])
ALPHA_META = read_json(ALPHA_RUN / "metadata.json")
ALL_FEATURES = read_json(ALPHA_RUN / "feature_names.json")

assert FEATURES and len(FEATURES) == len(set(FEATURES))
assert set(FEATURES).issubset(ALL_FEATURES)
assert MODEL_CONFIG["target_mode"] == "return", "本 notebook 实现收益回归"
assert SELECTION_META["selection_split"] == "train"
assert not SELECTION_META["validation_used_for_selection"]
assert not SELECTION_META["test_used_for_selection"]
assert SELECTION_META["target_name"] == MODEL_META["target_name"]

assert (
    file_hash(MODEL_DATA_RUN / "metadata.json")
    == SELECTION_SOURCE["model_metadata_sha256"]
)
assert (
    file_hash(MODEL_DATA_RUN / "config.json")
    == SELECTION_SOURCE["model_config_sha256"]
)
assert (
    file_hash(ALPHA_RUN / "metadata.json")
    == MODEL_META["source_metadata_sha256"]
)

# 当前正式产物为 Parquet。
assert MODEL_META["format"] == ALPHA_META["format"] == "parquet"

SPLITS = MODEL_CONFIG["splits"]
FEATURE_SELECTION_END = pd.Timestamp(
    SELECTION_SOURCE.get("model_config", MODEL_CONFIG)["splits"]["train"][1]
)

assert FEATURE_SELECTION_END < pd.Timestamp(SPLITS["valid"][0])

print("因子筛选版本：", SELECTION_RUN.name)
print("模型数据版本：", MODEL_DATA_RUN.name)
print("原始因子版本：", ALPHA_RUN.name)
print("已选因子数：", len(FEATURES))
print("预测标签：", MODEL_META["target_name"])

# 读取原始已选因子，以及第三模块审核过的标签
#
# 不直接使用已经标准化的 X：
# 在每次训练的 Pipeline 内重新拟合预处理，支持后面的滚动训练。

In [ ]:

INFO_COLUMNS = [
    "target",
    "y_raw",
    "label_end_date",
    "feature_eligible",
    "supervised_eligible",
]


def load_panel(splits=("train", "valid")):
    info_parts = []
    expected = pd.read_csv(
        MODEL_DATA_RUN / "split_summary.csv"
    ).set_index("split")

    for split in splits:
        paths = sorted((MODEL_DATA_RUN / "tree" / split).glob("part-*.parquet"))
        expected_parts = MODEL_META["partitions"]["tree"][split]

        if len(paths) != expected_parts:
            raise ValueError(f"{split} 数据分区缺失")

        split_parts = []

        for path in paths:
            frame = pd.read_parquet(path, columns=INFO_COLUMNS)

            if frame.index.names != ["datetime", "instrument"]:
                raise ValueError("数据索引必须为 datetime、instrument")

            for flag in ["feature_eligible", "supervised_eligible"]:
                if not frame[flag].isin([True, False]).all():
                    raise ValueError(f"{flag} 格式无效")

            # 保留缺标签但特征合格的行，供预测使用。
            frame = frame.loc[frame["feature_eligible"].eq(True)].copy()
            frame["label_end_date"] = pd.to_datetime(frame["label_end_date"])
            frame["split"] = split
            split_parts.append(frame)

        combined = pd.concat(split_parts)

        if combined["supervised_eligible"].sum() != expected.loc[
            split, "supervised_eligible"
        ]:
            raise ValueError(f"{split} 样本数与第三模块报告不符")

        start, end = map(pd.Timestamp, SPLITS[split])
        days = combined.index.get_level_values("datetime")
        assert ((days >= start) & (days <= end)).all()

        good = combined["supervised_eligible"].eq(True)
        assert combined.loc[good, "target"].notna().all()
        assert combined.loc[good, "label_end_date"].le(end).all()

        info_parts.append(combined)

    info = pd.concat(info_parts)

    if info.index.has_duplicates:
        raise ValueError("标签数据存在重复索引")

    pieces = []
    seen_instruments = set()
    paths = sorted(ALPHA_RUN.glob("part-*.parquet"))

    if len(paths) != ALPHA_META["partitions"]:
        raise ValueError("原始因子分区缺失")

    for path in paths:
        raw = pd.read_parquet(path, columns=FEATURES)
        instruments = set(raw.index.get_level_values("instrument"))

        if instruments & seen_instruments:
            raise ValueError("同一股票重复出现在原始因子分区中")

        seen_instruments.update(instruments)

        aligned = info.reindex(raw.index)
        mask = aligned["split"].notna()

        if mask.any():
            X = raw.loc[mask, FEATURES].replace(
                [np.inf, -np.inf], np.nan
            ).astype("float32")

            pieces.append(
                X.join(aligned.loc[mask], validate="one_to_one")
            )

    panel = pd.concat(pieces).sort_index()

    if panel.index.has_duplicates or len(panel) != len(info):
        raise ValueError("原始因子和标签无法完整一对一对齐")

    print("读取样本：", panel.groupby("split").size().to_dict())
    return panel


def sample_training(frame, max_rows=TRAIN_MAX_ROWS):
    """只在训练窗口内部抽样；所有模型共用相同索引。"""
    if frame.empty:
        raise ValueError("训练窗口没有可用样本")

    if max_rows is None or len(frame) <= max_rows:
        return frame.sort_index()

    hashes = pd.Series(
        pd.util.hash_pandas_object(frame.index, index=True).to_numpy(),
        index=frame.index,
    )

    return frame.loc[hashes.nsmallest(max_rows).index].sort_index()


splits_to_load = ("train", "valid", "test") if EVALUATE_TEST else ("train", "valid")
panel = load_panel(splits_to_load)

train_all = panel.loc[
    panel["split"].eq("train") & panel["supervised_eligible"].eq(True)
]
train = sample_training(train_all)

valid = panel.loc[panel["split"].eq("valid")]

assert train["label_end_date"].le(pd.Timestamp(SPLITS["train"][1])).all()

print(f"训练拟合样本：{len(train):,} / {len(train_all):,}")
print(f"验证特征样本：{len(valid):,}")
print(f"验证有标签样本：{valid['target'].notna().sum():,}")

# 训练内预处理和六模型工厂

In [ ]:
# 单元格 4：训练内预处理和六模型工厂

class TrainQuantileClipper(TransformerMixin, BaseEstimator):
    def __init__(self, quantiles=(0.01, 0.99)):
        self.quantiles = quantiles

    def fit(self, X, y=None):
        values = np.asarray(X, dtype=float)
        self.n_features_in_ = values.shape[1]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            self.lower_ = np.nanquantile(
                values, self.quantiles[0], axis=0
            )
            self.upper_ = np.nanquantile(
                values, self.quantiles[1], axis=0
            )

        self.lower_ = np.where(
            np.isfinite(self.lower_), self.lower_, -np.inf
        )
        self.upper_ = np.where(
            np.isfinite(self.upper_), self.upper_, np.inf
        )
        return self

    def transform(self, X):
        values = np.asarray(X, dtype=float)

        if values.shape[1] != self.n_features_in_:
            raise ValueError("预处理输入列数不一致")

        return np.clip(values, self.lower_, self.upper_)


def build_model(name, params=None):
    params = dict(params or {})

    if name == "ols":
        model = LinearRegression(
            **{
                "fit_intercept": True,
                "n_jobs": N_JOBS,
                **params,
            }
        )

    elif name == "lasso":
        model = Lasso(
            **{
                "alpha": 5e-5,
                "max_iter": 5000,
                "tol": 1e-5,
                "random_state": RANDOM_STATE,
                **params,
            }
        )

    elif name == "random_forest":
        model = RandomForestRegressor(
            **{
                "n_estimators": 120,
                "max_depth": 10,
                "min_samples_leaf": 100,
                "max_features": 0.8,
                "n_jobs": N_JOBS,
                "random_state": RANDOM_STATE,
                **params,
            }
        )

    elif name == "xgboost":
        model = xgb.XGBRegressor(
            **{
                "n_estimators": 300,
                "max_depth": 4,
                "learning_rate": 0.05,
                "subsample": 0.8,
                "colsample_bytree": 0.8,
                "reg_lambda": 5.0,
                "objective": "reg:squarederror",
                "tree_method": "hist",
                "n_jobs": N_JOBS,
                "random_state": RANDOM_STATE,
                **params,
            }
        )

    elif name == "lightgbm":
        model = lgb.LGBMRegressor(
            **{
                "n_estimators": 300,
                "num_leaves": 31,
                "learning_rate": 0.05,
                "min_child_samples": 100,
                "reg_lambda": 5.0,
                "n_jobs": N_JOBS,
                "random_state": RANDOM_STATE,
                "deterministic": True,
                "force_col_wise": True,
                "verbosity": -1,
                **params,
            }
        )

    elif name == "mlp":
        network = MLPRegressor(
            **{
                "hidden_layer_sizes": (64, 32),
                "alpha": 0.001,
                "batch_size": 1024,
                "learning_rate_init": 0.001,
                "max_iter": 150,
                "early_stopping": False,  # 不随机划出内部验证集
                "shuffle": True,         # 只打乱训练窗口内的批次
                "tol": 1e-4,
                "n_iter_no_change": 10,
                "random_state": RANDOM_STATE,
                **params,
            }
        )

        # 目标缩放只拟合训练 y，预测时自动还原为收益单位。
        model = TransformedTargetRegressor(
            regressor=network,
            transformer=StandardScaler(),
        )

    else:
        raise ValueError(f"未知模型：{name}")

    steps = []
    scaled_models = {"ols", "lasso", "mlp"}

    if name in scaled_models and CLIP_QUANTILES is not None:
        steps.append(
            ("clip", TrainQuantileClipper(CLIP_QUANTILES))
        )

    steps.append(
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                keep_empty_features=True,
            ),
        )
    )

    if name in scaled_models:
        steps.append(("scale", StandardScaler()))

    steps.append(("model", model))

    return Pipeline(steps)


def fit_model(name, params, train_frame, target_column="target"):
    estimator = build_model(name, params)
    start = time.perf_counter()

    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")

        with threadpool_limits(limits=N_JOBS):
            estimator.fit(
                train_frame[FEATURES],
                train_frame[target_column].to_numpy(dtype=float),
            )

    messages = sorted(set(str(w.message) for w in captured))

    return estimator, time.perf_counter() - start, messages


def predict_pipeline(estimator, X):
    pieces = []

    with threadpool_limits(limits=N_JOBS):
        for start in range(0, len(X), PREDICT_BATCH_ROWS):
            values = np.asarray(
                estimator.predict(
                    X.iloc[start:start + PREDICT_BATCH_ROWS]
                ),
                dtype=float,
            ).reshape(-1)

            if not np.isfinite(values).all():
                raise ValueError("模型预测存在 NaN 或无穷值")

            pieces.append(values)

    return np.concatenate(pieces) if pieces else np.array([])

# 统一评价、保存和加载接口

In [ ]:
# 单元格 5：统一评价、保存和加载接口

def corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    da, db = a - a.mean(), b - b.mean()
    denominator = np.sqrt(np.dot(da, da) * np.dot(db, db))

    if denominator <= 1e-15:
        return np.nan

    return float(np.clip(np.dot(da, db) / denominator, -1, 1))


def evaluate(y, predictions):
    frame = pd.DataFrame(
        {
            "label": y,
            "score": np.asarray(predictions, dtype=float),
        },
        index=y.index,
    ).replace([np.inf, -np.inf], np.nan).dropna()

    if frame.empty:
        raise ValueError("没有可评价的有标签预测")

    metrics = {
        "rows": len(frame),
        "rmse": float(
            np.sqrt(mean_squared_error(frame["label"], frame["score"]))
        ),
        "mae": float(
            mean_absolute_error(frame["label"], frame["score"])
        ),
        "r2": (
            float(r2_score(frame["label"], frame["score"]))
            if frame["label"].var() > 0 else np.nan
        ),
        "direction_accuracy": float(
            frame["label"].ge(0).eq(frame["score"].ge(0)).mean()
        ),
        "label_positive_rate": float(frame["label"].ge(0).mean()),
        "zero_forecast_rmse": float(
            np.sqrt(np.mean(frame["label"].to_numpy() ** 2))
        ),
    }

    records = []

    for day, block in frame.groupby(level="datetime", sort=True):
        ic = rank_ic = np.nan

        if len(block) >= MIN_CROSS_SECTION:
            score = block["score"].to_numpy()
            label = block["label"].to_numpy()

            ic = corr(score, label)
            rank_ic = corr(rankdata(score), rankdata(label))

        records.append({
            "datetime": day,
            "rows": len(block),
            "ic": ic,
            "rank_ic": rank_ic,
        })

    daily = pd.DataFrame(records)

    for field in ["ic", "rank_ic"]:
        values = daily[field].dropna()
        std = values.std(ddof=1)

        metrics[f"{field}_mean"] = float(values.mean())
        metrics[f"{field}_days"] = len(values)
        metrics[f"{field}_ir"] = (
            float(values.mean() / std)
            if std > 1e-15 else np.nan
        )

    # ICIR 为日均值/日标准差，不年化。
    return metrics, daily


def save_bundle(bundle, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("wb") as stream:
        cloudpickle.dump(bundle, stream, protocol=5)


def load_model(path, strict_versions=True):
    # 只加载自己生成并信任的文件。
    with Path(path).open("rb") as stream:
        bundle = cloudpickle.load(stream)

    if strict_versions:
        for package in ["sklearn", "xgboost", "lightgbm", "numpy"]:
            if bundle["versions"][package] != VERSIONS[package]:
                raise RuntimeError(f"{package} 版本与训练时不一致")

    return bundle


def predict_model(bundle, X, allow_historical=False):
    if list(X.columns) != bundle["features"]:
        raise ValueError("预测因子名称或顺序与训练时不一致")

    if X.index.names != ["datetime", "instrument"]:
        raise ValueError("预测索引必须为 datetime、instrument")

    if X.index.has_duplicates:
        raise ValueError("预测索引存在重复")

    if not allow_historical and len(X):
        days = X.index.get_level_values("datetime")

        if (days <= pd.Timestamp(bundle["available_after"])).any():
            raise ValueError(
                "预测日期早于模型可用时点；历史诊断请显式设置 "
                "allow_historical=True"
            )

    return pd.Series(
        predict_pipeline(bundle["pipeline"], X),
        index=X.index,
        name="score",
    )


def make_bundle(estimator, name, params, train_frame, available_after):
    return {
        "pipeline": estimator,
        "model": name,
        "params": params,
        "features": FEATURES.copy(),
        "versions": VERSIONS.copy(),
        "available_after": str(pd.Timestamp(available_after).date()),
        "train_rows": len(train_frame),
        "train_start": str(
            train_frame.index.get_level_values("datetime").min().date()
        ),
        "train_end": str(
            train_frame.index.get_level_values("datetime").max().date()
        ),
        "max_train_label_end": str(
            train_frame["label_end_date"].max().date()
        ),
        "train_index_sha256": hashlib.sha256(
            pd.util.hash_pandas_object(
                train_frame.index, index=True
            ).to_numpy().tobytes()
        ).hexdigest(),
        "model_data_run": str(MODEL_DATA_RUN),
        "selection_run": str(SELECTION_RUN),
    }


def save_predictions(frame, score, path, model, role, label="target"):
    output = pd.DataFrame(
        {
            "score": np.asarray(score, dtype=float),
            "label": frame[label],
            "label_end_date": frame["label_end_date"],
            "model": model,
            "role": role,
        },
        index=frame.index,
    )

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    output.to_parquet(path)

# 验证期用于选择参数，其结果不是最终无偏测试表现。
# 不在验证期重新筛因子，不使用测试数据选择参数。

In [ ]:
# 训练配置与公共接口：固定因子，三段时间验证，测试集不参与调参。
import copy
from sklearn.base import clone
from sklearn.model_selection import ParameterSampler
from IPython.display import display

EVALUATE_TEST = False
N_SEARCH = {"lasso": 8, "random_forest": 6, "xgboost": 8,
            "lightgbm": 8, "mlp": 6}
N_TIME_FOLDS = 3
STABILITY_WEIGHT = 0.25
EARLY_STOP_FRACTION = 0.15
EARLY_STOP_PATIENCE = 20
CV_TRAIN_MAX_ROWS = TRAIN_MAX_ROWS
FROZEN_FEATURES = tuple(FEATURES)
TRAINING_SCHEMA = "multi_model_training_v1"

SEARCH_SPACES = {
    "lasso": {"alpha": np.logspace(-5, -3, 8).tolist(),
              "max_iter": [10000], "tol": [1e-5]},
    "random_forest": {"n_estimators": [200], "max_depth": [4, 6, 8],
                      "min_samples_leaf": [100, 300], "max_features": [0.7, 1.0],
                      "max_samples": [0.7], "bootstrap": [True]},
    "xgboost": {"n_estimators": [1000], "learning_rate": [0.03, 0.05],
                "max_depth": [2, 3, 4], "min_child_weight": [20, 50],
                "subsample": [0.8], "colsample_bytree": [0.8],
                "reg_alpha": [0.0, 0.1], "reg_lambda": [5.0, 20.0, 50.0],
                "gamma": [0.0, 0.1]},
    "lightgbm": {"n_estimators": [1000], "learning_rate": [0.03, 0.05],
                 "max_depth": [4, 5], "num_leaves": [7, 15],
                 "min_child_samples": [100, 300], "subsample": [0.8],
                 "subsample_freq": [1], "colsample_bytree": [0.8],
                 "reg_alpha": [0.0, 0.1], "reg_lambda": [5.0, 20.0, 50.0]},
    "mlp": {"hidden_layer_sizes": [(32,), (64,), (32, 16)],
            "activation": ["relu"], "alpha": [0.001, 0.01, 0.1],
            "learning_rate_init": [0.0005, 0.001], "batch_size": [1024],
            "max_iter": [200], "early_stopping": [False],
            "tol": [1e-5], "n_iter_no_change": [20]},
}


def json_ready(value):
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (tuple, list, np.ndarray)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.generic):
        return json_ready(value.item())
    if isinstance(value, (pd.Timestamp, Path)):
        return str(value)
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(json_ready(value), ensure_ascii=False,
                                    indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def object_hash(value):
    text = json.dumps(json_ready(value), sort_keys=True, allow_nan=False)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def clean_params(params):
    return {k: v.item() if isinstance(v, np.generic) else v
            for k, v in params.items()}


def panel_dates(frame):
    return pd.DatetimeIndex(frame.index.get_level_values("datetime"))


def training_source():
    """绑定因子列表和上游版本，防止恢复/滚动时混用其他版本。"""
    if tuple(FEATURES) != FROZEN_FEATURES:
        raise ValueError("FEATURES 已变化；请重新运行版本绑定和本配置格。")
    if read_json(SELECTION_RUN / "selected_factors.json") != list(FROZEN_FEATURES):
        raise ValueError("磁盘中的因子列表与本次训练不一致。")
    expected = {"model_metadata_sha256": MODEL_DATA_RUN / "metadata.json",
                "model_config_sha256": MODEL_DATA_RUN / "config.json"}
    for key, path in expected.items():
        if file_hash(path) != SELECTION_SOURCE[key]:
            raise ValueError(f"上游版本校验失败：{key}")
    if file_hash(ALPHA_RUN / "metadata.json") != MODEL_META["source_metadata_sha256"]:
        raise ValueError("原始因子版本已变化。")
    return {"model_data_run": str(MODEL_DATA_RUN),
            "selection_run": str(SELECTION_RUN), "alpha_run": str(ALPHA_RUN),
            "model_metadata_sha256": SELECTION_SOURCE["model_metadata_sha256"],
            "model_config_sha256": SELECTION_SOURCE["model_config_sha256"],
            "selection_sha256": file_hash(SELECTION_RUN / "selected_factors.json"),
            "selection_metadata_sha256": file_hash(SELECTION_RUN / "metadata.json"),
            "selection_source_sha256": file_hash(SELECTION_RUN / "source.json"),
            "alpha_metadata_sha256": MODEL_META["source_metadata_sha256"],
            "features": list(FROZEN_FEATURES)}


def evaluate_model(y, score):
    """沿用原评价函数，汇总字段统一使用 rank_ic_mean。"""
    metrics, daily = evaluate(y, score)
    required = {"rmse", "zero_forecast_rmse", "rank_ic_mean", "ic_mean"}
    missing = required.difference(metrics)
    if missing:
        raise KeyError(f"评价接口缺少字段：{sorted(missing)}")
    baseline = float(metrics["zero_forecast_rmse"])
    if not np.isfinite(baseline) or baseline <= 0:
        raise ValueError("零收益预测基准 RMSE 无效。")
    metrics["relative_rmse"] = float(metrics["rmse"] / baseline)
    if not np.isfinite(metrics["relative_rmse"]):
        raise ValueError("相对 RMSE 无效。")
    return metrics, daily


def split_training_for_early_stop(frame):
    days = panel_dates(frame)
    unique = days.unique().sort_values()
    if len(unique) < 20:
        raise ValueError("训练日期不足 20 个交易日。")
    position = min(max(int(len(unique) * (1 - EARLY_STOP_FRACTION)), 1), len(unique)-1)
    stop_start = unique[position]
    ends = pd.to_datetime(frame["label_end_date"])
    fit = frame.loc[(days < stop_start) & (ends < stop_start)].copy()
    stop = frame.loc[(days >= stop_start) & frame["target"].notna()].copy()
    if fit.empty or stop.empty:
        raise ValueError("内部训练集或早停集为空。")
    return fit, stop


def choose_training_length(name, params, frame):
    fit, stop = split_training_for_early_stop(frame)
    pipeline = build_model(name, params)
    preprocess = Pipeline(pipeline.steps[:-1])
    X_fit = preprocess.fit_transform(fit[FEATURES])
    X_stop = preprocess.transform(stop[FEATURES])
    y_fit = fit["target"].to_numpy(dtype=float)
    y_stop = stop["target"].to_numpy(dtype=float)
    estimator = clone(pipeline.steps[-1][1])
    with threadpool_limits(limits=N_JOBS):
        if name == "xgboost":
            estimator.set_params(early_stopping_rounds=EARLY_STOP_PATIENCE,
                                 eval_metric="rmse")
            estimator.fit(X_fit, y_fit, eval_set=[(X_stop, y_stop)], verbose=False)
            return int(estimator.best_iteration) + 1
        if name == "lightgbm":
            estimator.fit(X_fit, y_fit, eval_set=[(X_stop, y_stop)], eval_metric="rmse",
                          callbacks=[lgb.early_stopping(EARLY_STOP_PATIENCE,
                                     first_metric_only=True, verbose=False),
                                     lgb.log_evaluation(0)])
            return int(estimator.best_iteration_ or params["n_estimators"])
        if name == "mlp":
            network = clone(estimator.regressor)
            scaler = StandardScaler()
            fit_y = scaler.fit_transform(y_fit.reshape(-1, 1)).ravel()
            stop_y = scaler.transform(y_stop.reshape(-1, 1)).ravel()
            best_loss, best_epoch, stale = np.inf, 1, 0
            for epoch in range(1, int(params["max_iter"]) + 1):
                network.partial_fit(X_fit, fit_y)
                loss = float(np.mean((stop_y - network.predict(X_stop))**2))
                if not np.isfinite(loss):
                    raise ValueError("MLP 内部早停损失无效。")
                if loss < best_loss - 1e-5:
                    best_loss, best_epoch, stale = loss, epoch, 0
                else:
                    stale += 1
                if stale >= EARLY_STOP_PATIENCE:
                    break
            return best_epoch
    raise ValueError(f"不支持内部早停的模型：{name}")


def fixed_length_params(name, params, length):
    effective = clean_params(copy.deepcopy(params))
    if name in {"xgboost", "lightgbm"}:
        effective["n_estimators"] = int(length)
    elif name == "mlp":
        effective.update(max_iter=int(length), early_stopping=False,
                         tol=0.0, n_iter_no_change=int(length)+1)
    return effective


def fit_with_training_early_stop(name, params, frame):
    started = time.perf_counter()
    effective = clean_params(copy.deepcopy(params))
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        if name in {"xgboost", "lightgbm", "mlp"}:
            length = choose_training_length(name, effective, frame)
            effective = fixed_length_params(name, effective, length)
        estimator, _, messages = fit_model(name, effective, frame)
    messages = sorted(set(messages + [str(w.message) for w in captured]))
    return estimator, effective, time.perf_counter()-started, messages


def make_time_folds(data, n_folds=3, max_rows=CV_TRAIN_MAX_ROWS):
    days = panel_dates(data)
    ends = pd.to_datetime(data["label_end_date"])
    valid_days = panel_dates(data.loc[data["split"].eq("valid")]).unique().sort_values()
    if n_folds < 2 or len(valid_days) < n_folds:
        raise ValueError("时间验证至少需要两个非空窗口。")
    if pd.Timestamp(FEATURE_SELECTION_END) >= valid_days.min():
        raise ValueError("因子筛选日期覆盖了验证期。")
    folds, summary = [], []
    for fold, block in enumerate(np.array_split(valid_days, n_folds)):
        start, end = pd.Timestamp(block[0]), pd.Timestamp(block[-1])
        fit_mask = ((days >= pd.Timestamp(SPLITS["train"][0])) & (days < start)
                    & data["y_raw"].notna() & (ends < start))
        val_mask = ((days >= start) & (days <= end) & data["y_raw"].notna()
                    & (ends <= end))
        fit, val = data.loc[fit_mask].copy(), data.loc[val_mask].copy()
        fit["target"], val["target"] = fit["y_raw"], val["y_raw"]
        fit = sample_training(fit, max_rows=max_rows)
        if val.empty or fit["label_end_date"].max() >= start:
            raise ValueError(f"fold={fold} 为空或存在跨边界训练标签。")
        folds.append((fold, fit, val))
        summary.append(dict(fold=fold, valid_start=str(start.date()),
                            valid_end=str(end.date()), train_rows=len(fit),
                            valid_rows=len(val), max_train_label_end=str(fit.label_end_date.max())))
    return folds, pd.DataFrame(summary)


def candidate_list(name, config):
    budget = int(config["n_search"][name]) if config["run_tuning"] else 1
    if budget < 1:
        raise ValueError("参数搜索次数必须为正数。")
    candidates = list(ParameterSampler(config["search_spaces"][name],
                       n_iter=budget, random_state=config["random_state"]))
    return [clean_params(p) for p in candidates]


def tune_model(name, folds, config, run_dir=None):
    """统一调参接口，返回冻结的参数、候选汇总和逐窗口记录。"""
    if name == "ols":
        return {}, pd.DataFrame(), pd.DataFrame(), np.nan
    trials, records, best_rows = [], [], []
    best_params, best_objective = None, np.inf
    candidates = candidate_list(name, config)
    for trial, params in enumerate(candidates, 1):
        print(f"\n{name} 参数组 {trial}/{len(candidates)} {params}", flush=True)
        rows, failure = [], None
        started = time.perf_counter()
        for fold, fit, val in folds:
            try:
                estimator, effective, seconds, messages = fit_with_training_early_stop(name, params, fit)
                score = predict_pipeline(estimator, val[FEATURES])
                metrics, _ = evaluate_model(val["target"], score)
                row = dict(model=name, trial=trial, fold=fold, **metrics,
                           fit_seconds=seconds, effective_params=json_ready(effective),
                           warnings=messages, status="success")
                rows.append(row)
                records.append(row)
                print(f"  fold={fold} RMSE={metrics['rmse']:.6f} "
                      f"相对RMSE={metrics['relative_rmse']:.4f} "
                      f"RankIC={metrics['rank_ic_mean']:.5f} 耗时={seconds:.1f}s "
                      f"有效轮数={effective.get('n_estimators', effective.get('max_iter', '-'))}", flush=True)
                del estimator
            except Exception as exc:
                failure = f"{type(exc).__name__}: {exc}"
                records.append(dict(model=name, trial=trial, fold=fold,
                                    status="failed", error=failure))
                print(f"  参数组失败：{failure}", flush=True)
                break
            finally:
                if run_dir is not None:
                    pd.DataFrame(records).to_csv(Path(run_dir)/f"cv_{name}.csv", index=False)
        result = dict(model=name, trial=trial, params=json_ready(params),
                      seconds=time.perf_counter()-started,
                      status="failed" if failure else "success", error=failure)
        if failure is None:
            values = np.array([r["relative_rmse"] for r in rows])
            objective = float(values.mean()+config["stability_weight"]*values.std(ddof=1))
            result.update(objective=objective, mean_relative_rmse=float(values.mean()),
                          std_relative_rmse=float(values.std(ddof=1)),
                          rank_ic_mean=float(pd.Series([r["rank_ic_mean"] for r in rows]).mean()))
            if objective < best_objective:
                best_params, best_objective, best_rows = copy.deepcopy(params), objective, rows
        trials.append(result)
        if run_dir is not None:
            pd.DataFrame(trials).to_csv(Path(run_dir)/f"trials_{name}.csv", index=False)
    if best_params is None:
        raise RuntimeError(f"{name} 所有候选均失败，请检查 cv_{name}.csv。")
    if name in {"xgboost", "lightgbm", "mlp"}:
        key = "max_iter" if name == "mlp" else "n_estimators"
        length = int(np.median([r["effective_params"][key] for r in best_rows]))
        best_params = fixed_length_params(name, best_params, length)
    return best_params, pd.DataFrame(trials), pd.DataFrame(records), best_objective


def default_training_config():
    return dict(schema=TRAINING_SCHEMA, models=list(MODEL_NAMES),
                n_search=copy.deepcopy(N_SEARCH), search_spaces=copy.deepcopy(SEARCH_SPACES),
                time_folds=N_TIME_FOLDS, fit_max_rows=TRAIN_MAX_ROWS,
                cv_max_rows=CV_TRAIN_MAX_ROWS, random_state=RANDOM_STATE,
                n_jobs=N_JOBS, run_tuning=bool(RUN_TUNING),
                stability_weight=STABILITY_WEIGHT, early_stop_fraction=EARLY_STOP_FRACTION,
                early_stop_patience=EARLY_STOP_PATIENCE, clip_quantiles=CLIP_QUANTILES,
                min_cross_section=MIN_CROSS_SECTION, splits=copy.deepcopy(SPLITS),
                selection_metric="mean_relative_rmse_plus_stability_penalty",
                test_used=False, final_fit="original_train_split")


def save_verified_model(bundle, path, probe):
    """统一保存接口：模型文件、JSON 摘要、预测往返校验。"""
    path = Path(path)
    before = predict_model(bundle, probe, allow_historical=True)
    save_bundle(bundle, path)
    restored = load_model(path)
    after = predict_model(restored, probe, allow_historical=True)
    np.testing.assert_allclose(before, after, rtol=1e-10, atol=1e-12)
    header = {k: v for k, v in bundle.items() if k != "pipeline"}
    header["model_sha256"] = file_hash(path)
    atomic_json(path.with_suffix(".json"), header)
    return header["model_sha256"]


def run_training(data, run_dir=None, config=None):
    """统一训练接口；可传入未完成的 run_dir，跳过已完成的模型。"""
    source = training_source()
    config = copy.deepcopy(config or default_training_config())
    config["source"] = source
    config["versions"] = VERSIONS.copy()
    if config.get("test_used") or config["splits"] != SPLITS:
        raise ValueError("训练配置包含测试调参或与当前上游时间区间不一致。")
    if set(config["models"]) - set(MODEL_NAMES) or not config["models"]:
        raise ValueError("训练模型名称无效。")
    # 即使调用者传了测试分区，也禁止本接口使用。
    if set(data["split"].dropna()) - {"train", "valid"}:
        raise ValueError("run_training 只接受 train/valid 分区。")
    if data.index.has_duplicates:
        raise ValueError("训练数据索引重复。")
    fingerprint = object_hash(config)
    if run_dir is None:
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        run_dir = OUTPUT_ROOT/f"run_{stamp}_{uuid.uuid4().hex[:8]}"
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    for folder in ("models", "predictions", "daily_metrics", "completed"):
        (run_dir/folder).mkdir(exist_ok=True)
    if (run_dir/"config.json").exists():
        if object_hash(read_json(run_dir/"config.json")) != fingerprint:
            raise ValueError("恢复目录的配置/上游版本不一致，请新建运行。")
    else:
        atomic_json(run_dir/"config.json", config)
    metadata = dict(schema=TRAINING_SCHEMA, status="running", run_id=run_dir.name,
                    models=config["models"], features=list(FROZEN_FEATURES), source=source,
                    versions=VERSIONS.copy(), config_sha256=fingerprint,
                    feature_selection_end=str(pd.Timestamp(FEATURE_SELECTION_END).date()),
                    tuning_end=str(SPLITS["valid"][1]), evaluation_role="tuning_validation",
                    test_evaluated=False, selection_metric=config["selection_metric"],
                    **{k: source[k] for k in ("model_metadata_sha256", "model_config_sha256", "selection_sha256")})
    atomic_json(run_dir/"metadata.json", metadata)
    fit_all = data.loc[data["split"].eq("train") & data["supervised_eligible"].eq(True)].copy()
    static_train = sample_training(fit_all, max_rows=config["fit_max_rows"])
    validation = data.loc[data["split"].eq("valid")]
    if validation.empty or static_train["target"].isna().any():
        raise ValueError("训练或验证数据无效。")
    folds, fold_summary = make_time_folds(data, config["time_folds"], config["cv_max_rows"])
    fold_summary.to_csv(run_dir/"cv_windows.csv", index=False)
    display(fold_summary)
    best, comparison, inventory, all_trials, all_records = {}, [], {}, [], []
    print("本次目录：", run_dir, flush=True)
    for name in config["models"]:
        marker = run_dir/"completed"/f"{name}.json"
        if marker.exists():
            completed = read_json(marker)
            if completed["config_sha256"] != fingerprint:
                raise ValueError("已完成模型的配置不一致。")
            for relative, digest in completed["files"].items():
                if file_hash(run_dir/relative) != digest:
                    raise ValueError(f"已完成产物校验失败：{relative}")
            best[name], inventory[name] = completed["params"], completed["model_sha256"]
            comparison.append(completed["result"])
            for filename, target in ((f"trials_{name}.csv", all_trials), (f"cv_{name}.csv", all_records)):
                if (run_dir/filename).exists():
                    target.append(pd.read_csv(run_dir/filename))
            print(f"{name} 已完成并校验，跳过重复训练。", flush=True)
            continue
        print(f"\n{'='*48}\n开始训练：{name}", flush=True)
        params, trials, records, objective = tune_model(name, folds, config, run_dir)
        if not trials.empty:
            all_trials.append(trials)
            all_records.append(records)
        best[name] = params
        print(f"{name} 最终参数：{params}", flush=True)
        # OLS 和其他模型均沿用原训练期静态拟合；OLS 工厂保持不变。
        estimator, seconds, messages = fit_model(name, params, static_train)
        score = predict_pipeline(estimator, validation[FEATURES])
        metrics, daily = evaluate_model(validation["target"], score)
        bundle = make_bundle(estimator, name, params, static_train, SPLITS["valid"][1])
        bundle.update(schema=TRAINING_SCHEMA, source=source, config_sha256=fingerprint,
                      evaluation_role="tuning_validation", test_used_for_tuning=False,
                      fit_warnings=messages, features_frozen=True)
        model_path = run_dir/"models"/f"{name}.pkl"
        digest = save_verified_model(bundle, model_path, validation[FEATURES].iloc[:100])
        inventory[name] = digest
        pred_path = run_dir/"predictions"/f"{name}_valid.parquet"
        save_predictions(validation, score, pred_path, name, "tuning_validation")
        daily_path = run_dir/"daily_metrics"/f"{name}_valid.csv"
        daily.to_csv(daily_path, index=False)
        result = dict(model=name, evaluation_role="tuning_validation", fit_seconds=seconds,
                      cv_objective=objective, fit_warnings=" | ".join(messages),
                      **{f"valid_{k}": v for k, v in metrics.items()})
        if not trials.empty:
            winner = trials.loc[trials["status"].eq("success")].sort_values("objective").iloc[0]
            result.update(cv_relative_rmse=winner["mean_relative_rmse"],
                          cv_rank_ic_mean=winner["rank_ic_mean"])
        comparison.append(result)
        tracked = [model_path, model_path.with_suffix(".json"), pred_path, daily_path]
        tracked += [run_dir/fn for fn in (f"trials_{name}.csv", f"cv_{name}.csv") if (run_dir/fn).exists()]
        atomic_json(marker, dict(config_sha256=fingerprint, params=params, result=result,
                                model_sha256=digest, files={str(p.relative_to(run_dir)): file_hash(p) for p in tracked}))
        atomic_json(run_dir/"best_params.json", best)
        pd.DataFrame(comparison).to_csv(run_dir/"model_comparison.csv", index=False)
        print(f"{name} 已保存且加载校验通过：RMSE={metrics['rmse']:.6f} "
              f"RankIC={metrics['rank_ic_mean']:.5f}，以上为调参诊断。", flush=True)
    if all_trials:
        pd.concat(all_trials, ignore_index=True).to_csv(run_dir/"tuning_trials.csv", index=False)
        pd.concat(all_records, ignore_index=True).to_csv(run_dir/"cv_fold_metrics.csv", index=False)
    atomic_json(run_dir/"best_params.json", best)
    pd.DataFrame(comparison).to_csv(run_dir/"model_comparison.csv", index=False)
    metadata.update(status="complete", completed_at=datetime.now(timezone.utc).isoformat(),
                    model_sha256=inventory, best_params_sha256=file_hash(run_dir/"best_params.json"))
    atomic_json(run_dir/"metadata.json", metadata)
    if run_dir.parent.resolve() == OUTPUT_ROOT.resolve():
        atomic_json(OUTPUT_ROOT/"latest_run.json", {"run_dir": run_dir.name})
    return run_dir, best, pd.DataFrame(comparison)


print("搜索预算：", N_SEARCH)
print("外层搜索拟合次数：", sum(N_SEARCH.values())*N_TIME_FOLDS,
      "；另有早停探测和最终拟合；OLS 保持原方式。")
print("汇总指标统一使用 rank_ic_mean；因子固定，测试集不参与调参。")


In [ ]:
# 正式训练：运行本格开始；前面已读取 panel 时不会再读一遍全量数据。
# 若中断后继续：把 RESUME_TRAIN_RUN 改为打印出来的 run_... 目录。
# 恢复时跳过已完整保存且校验通过的模型；未完成模型重新搜索。
RUN_TRAINING = True
RESUME_TRAIN_RUN = None  # 例如 OUTPUT_ROOT / "run_2026..."

if RUN_TRAINING:
    if "panel" not in globals() or set(panel["split"].dropna()) != {"train", "valid"}:
        panel = load_panel(splits=("train", "valid"))
    RUN_DIR, best_params, comparison = run_training(panel, run_dir=RESUME_TRAIN_RUN)
    display(comparison)
    print("六模型训练完成：", RUN_DIR)
    print("以上为验证期调参诊断，尚未运行独立测试或滚动训练。")
else:
    print("训练未执行；需要时把 RUN_TRAINING 改为 True。")


## 6. 统一加载、预测与评价

上面的两个代码格已修复 `rank_ic` / `rank_ic_mean` 不一致。前面的模型工厂、OLS 和评价函数保持原样。

- `run_training`：训练六模型并调参；`tune_model`：单模型的时间验证搜索。
- `save_verified_model`：保存 Pipeline、预处理参数和版本信息，并检查加载前后的预测一致。
- `load_training_run`：加载一整组模型，校验参数、文件及因子版本。
- `predict_models`：按相同索引批量预测；`evaluate_models`：统一误差、IC 和 RankIC。

因子固定使用第四模块的筛选结果。2024 年划为三个连续时间验证窗口；后续窗口使用此前已实现的标签扩展训练历史，每个窗口最多抽取 10 万训练样本。预处理及早停只拟合当前训练窗口。最终静态模型仍拟合原训练期，OLS 不改变。

默认选参目标是平均相对 RMSE 加 0.25 倍窗口标准差；第一格旧的 `SELECTION_METRIC` / `PARAM_GRIDS` 不再驱动本版搜索，以 `config.json` 为准。验证结果属于调参诊断。测试与滚动开关默认关闭。

In [ ]:
# 统一加载、批量预测和模型评价；只使用保存的 Pipeline，不重新拟合。
def load_training_run(run_dir=None, check_context=True):
    if run_dir is None:
        run_dir = latest_run(OUTPUT_ROOT)
    run_dir = Path(run_dir)
    metadata = read_json(run_dir/"metadata.json")
    if metadata.get("schema") != TRAINING_SCHEMA or metadata.get("status") != "complete":
        raise ValueError("请选择本版代码生成且六模型训练成功的运行目录。")
    config = read_json(run_dir/"config.json")
    if object_hash(config) != metadata["config_sha256"]:
        raise ValueError("配置文件校验失败。")
    if check_context and metadata["source"] != training_source():
        raise ValueError("当前 notebook 与保存模型的因子/数据版本不一致。")
    if file_hash(run_dir/"best_params.json") != metadata["best_params_sha256"]:
        raise ValueError("冻结参数文件校验失败。")
    params = read_json(run_dir/"best_params.json")
    bundles = {}
    for name in metadata["models"]:
        path = run_dir/"models"/f"{name}.pkl"
        if file_hash(path) != metadata["model_sha256"][name]:
            raise ValueError(f"模型文件校验失败：{name}")
        # 仅加载自己生成且信任的 cloudpickle 文件。
        bundle = load_model(path)
        if (bundle["features"] != metadata["features"]
                or object_hash(bundle["params"]) != object_hash(params[name])
                or bundle["source"] != metadata["source"]
                or bundle["config_sha256"] != metadata["config_sha256"]):
            raise ValueError(f"模型与运行清单不一致：{name}")
        bundles[name] = bundle
    return dict(run_dir=run_dir, metadata=metadata, config=config,
                best_params=params, bundles=bundles)


def predict_models(loaded, X, output_path=None, allow_historical=False):
    """返回索引为 (datetime,instrument)、列为模型名的预测矩阵。"""
    expected = loaded["metadata"]["features"]
    if list(X.columns) != expected:
        raise ValueError("请按保存的因子顺序提供 X，不能静默丢列或添加列。")
    scores = pd.DataFrame(index=X.index)
    for name, bundle in loaded["bundles"].items():
        scores[name] = predict_model(bundle, X, allow_historical=allow_historical)
    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        scores.to_parquet(output_path)
    return scores


def evaluate_models(y, scores):
    """统一评价接口；日期横截面 IC 使用原评价函数的口径。"""
    if not y.index.equals(scores.index):
        raise ValueError("标签和预测索引必须完全一致。")
    records, daily = [], {}
    for name in scores:
        metrics, daily[name] = evaluate_model(y, scores[name].to_numpy())
        records.append(dict(model=name, **metrics))
    return pd.DataFrame(records), daily


def evaluate_static_test(loaded, test_data=None, output_dir=None):
    """参数冻结后的独立测试；不据此调参或重新筛因子。"""
    if test_data is None:
        test_data = load_panel(splits=("test",))
    if set(test_data["split"].dropna()) != {"test"}:
        raise ValueError("独立测试只接受 test 分区。")
    scores = predict_models(loaded, test_data[FEATURES])
    labeled = test_data["target"].notna()
    report, daily = evaluate_models(test_data.loc[labeled, "target"], scores.loc[labeled])
    report["evaluation_role"] = "static_holdout_test"
    output_dir = Path(output_dir or loaded["run_dir"]/"test_evaluation")
    output_dir.mkdir(parents=True, exist_ok=True)
    scores.to_parquet(output_dir/"scores.parquet")
    report.to_csv(output_dir/"model_comparison.csv", index=False)
    for name, table in daily.items():
        table.to_csv(output_dir/f"{name}_daily.csv", index=False)
    atomic_json(output_dir/"metadata.json", dict(base_run=str(loaded["run_dir"]),
                role="static_holdout_test", parameters_frozen=True, source=loaded["metadata"]["source"]))
    return report, scores


print("接口已就绪：run_training / tune_model / save_verified_model / load_training_run")
print("预测与评价：predict_models / evaluate_models / evaluate_static_test")

### 加载示例与验证期诊断

正式训练成功后运行。重新打开 notebook 时，可把 `MODEL_RUN_TO_LOAD` 指定为已完成的 `run_...` 路径；留空使用最新成功版本。这里对验证期的预测仅用于接口检查，显式允许历史诊断。

In [ ]:
MODEL_RUN_TO_LOAD = globals().get("RUN_DIR", None)
loaded_models = load_training_run(MODEL_RUN_TO_LOAD)
display(pd.DataFrame([
    {"model": name, "params": params}
    for name, params in loaded_models["best_params"].items()
]))
print("已加载：", loaded_models["run_dir"])

# 少量历史样本检查统一预测接口；不在这里重新训练。
if "valid" in globals() and len(valid):
    prediction_example = predict_models(
        loaded_models, valid[FEATURES].iloc[:100], allow_historical=True
    )
    display(prediction_example.head())

validation_report = pd.read_csv(loaded_models["run_dir"] / "model_comparison.csv")
display(validation_report)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
validation_report.plot.bar(x="model", y="valid_relative_rmse", ax=axes[0], legend=False)
axes[0].axhline(1, color="gray", linestyle="--")
axes[0].set_title("Tuning validation: relative RMSE")
validation_report.plot.bar(x="model", y="valid_rank_ic_mean", ax=axes[1], legend=False)
axes[1].axhline(0, color="gray", linestyle="--")
axes[1].set_title("Tuning validation: mean RankIC")
plt.tight_layout()
plt.show()

## 7. 可选：冻结参数后的独立测试

只在因子、搜索空间、选参标准和训练方案都确定后开启。此步骤不拟合模型、不筛因子、不选择参数；预测包括没有标签的行，评价只使用有效标签。看过测试结果后修改方案，不能再把同一时期视为未使用的测试集。

In [ ]:
RUN_STATIC_TEST = False

if RUN_STATIC_TEST:
    static_test_report, static_test_scores = evaluate_static_test(loaded_models)
    display(static_test_report)
else:
    print("独立测试未执行：RUN_STATIC_TEST=False。")

## 8. 固定因子与超参数的滚动训练

默认从 2025 年开始，每三个月重新拟合一次；`train_months=None` 使用扩展历史，`24` 则只使用最近 24 个月。每次只允许 `label_end_date < 窗口起点` 的样本参与训练。原始标签 `y_raw` 用于恢复此前跨分区、但当时已经实现的标签。

滚动阶段固定因子、超参数、树数和 MLP epoch，仅重新拟合预处理和模型系数/树结构/网络权重。预测期表现不会反馈到选参。每折模型、预测、标签日期、质量指标和完整版本单独保存；含缺标签的末端预测也保留。预测得分可供后续回测模块使用，此处不计算持仓、费用或净值。

In [ ]:
# 滚动训练：只更新模型拟合参数和训练内预处理；因子和超参数保持冻结。
def rolling_windows(start, end, retrain_months=3):
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    if retrain_months < 1 or start > end:
        raise ValueError("滚动时间区间或重训间隔无效。")
    windows, cursor = [], start
    while cursor <= end:
        next_start = cursor + pd.DateOffset(months=int(retrain_months))
        windows.append((cursor, min(end, next_start-pd.Timedelta(days=1))))
        cursor = next_start
    return windows


def run_rolling(base_run=None, start=None, end=None, retrain_months=3,
                train_months=None, max_train_rows=TRAIN_MAX_ROWS,
                model_names=None, data=None, output_root=None):
    """
    train_months=None：扩展历史窗口；如 24：最近 24 个月的移动窗口。
    每个预测窗口开始前，只使用 label_end_date < 窗口起点的标签。
    不重新调参，不使用本次预测窗口的表现决定模型参数。
    """
    loaded = load_training_run(base_run)
    frozen_params = copy.deepcopy(loaded["best_params"])
    frozen_hash = object_hash(frozen_params)
    source = loaded["metadata"]["source"]
    start = pd.Timestamp(start or SPLITS["test"][0])
    end = pd.Timestamp(end or SPLITS["test"][1])
    names = list(model_names or loaded["metadata"]["models"])
    if not names or len(names) != len(set(names)) or set(names)-set(frozen_params):
        raise ValueError("滚动训练模型列表无效。")
    if train_months is not None and int(train_months) < 1:
        raise ValueError("train_months 必须为正数或 None。")
    available = max(pd.Timestamp(loaded["metadata"]["tuning_end"]),
                    pd.Timestamp(loaded["metadata"]["feature_selection_end"]))
    if start <= available:
        raise ValueError("滚动起点必须晚于因子筛选和参数选择的截止时间。")
    if start < pd.Timestamp(SPLITS["test"][0]) or end > pd.Timestamp(SPLITS["test"][1]):
        raise ValueError("当前研究数据的滚动预测区间必须位于 test 区间内。")
    if data is None:
        data = load_panel(splits=("train", "valid", "test"))
    if data.index.has_duplicates:
        raise ValueError("滚动面板索引重复。")
    # 验证上游版本；不会因 latest 指针变化静默换一组因子。
    if source != training_source():
        raise ValueError("滚动训练与静态训练的数据/因子版本不一致。")
    days = panel_dates(data)
    ends = pd.to_datetime(data["label_end_date"])
    windows = rolling_windows(start, end, retrain_months)
    output_root = Path(output_root or OUTPUT_ROOT/"rolling")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_dir = output_root/f"run_{stamp}_{uuid.uuid4().hex[:8]}"
    run_dir.mkdir(parents=True, exist_ok=False)
    config = dict(base_run=str(loaded["run_dir"]), start=str(start.date()), end=str(end.date()),
                  retrain_months=retrain_months, train_months=train_months,
                  max_train_rows=max_train_rows, models=names, best_params=frozen_params,
                  frozen_params_sha256=frozen_hash, source=source, versions=VERSIONS,
                  role="walk_forward_test", label_availability="label_end_date < fold_start")
    atomic_json(run_dir/"config.json", config)
    atomic_json(run_dir/"metadata.json", dict(status="running", **config))
    metrics_rows, window_rows, score_paths = [], [], {name: [] for name in names}
    for fold, (fold_start, fold_end) in enumerate(windows):
        history_start = pd.Timestamp(SPLITS["train"][0])
        if train_months is not None:
            history_start = max(history_start, fold_start-pd.DateOffset(months=int(train_months)))
        fit_mask = ((days >= history_start) & (days < fold_start)
                    & data["y_raw"].notna() & (ends < fold_start))
        pred_mask = (days >= fold_start) & (days <= fold_end)
        predict_frame = data.loc[pred_mask].copy()
        if predict_frame.empty:
            raise ValueError(f"滚动窗口 {fold} 没有特征数据，检查预测区间。")
        fit = sample_training(data.loc[fit_mask].copy(), max_rows=max_train_rows)
        fit["target"] = fit["y_raw"]
        if fit.label_end_date.max() >= fold_start:
            raise ValueError("滚动训练标签尚未在窗口开始前实现。")
        window_rows.append(dict(fold=fold, predict_start=str(fold_start.date()),
                                predict_end=str(fold_end.date()), train_rows=len(fit),
                                train_start=str(panel_dates(fit).min().date()),
                                train_end=str(panel_dates(fit).max().date()),
                                max_train_label_end=str(fit.label_end_date.max().date()),
                                predict_rows=len(predict_frame)))
        print(f"\n滚动窗口 {fold+1}/{len(windows)}：{fold_start.date()} ~ {fold_end.date()}，"
              f"拟合 {len(fit):,} 行，预测 {len(predict_frame):,} 行", flush=True)
        fold_dir = run_dir/f"fold-{fold:03d}"
        (fold_dir/"models").mkdir(parents=True)
        for name in names:
            # 已冻结训练轮数，滚动时直接拟合，不再次早停/调参。
            estimator, seconds, messages = fit_model(name, frozen_params[name], fit)
            score = predict_pipeline(estimator, predict_frame[FEATURES])
            bundle = make_bundle(estimator, name, frozen_params[name], fit,
                                 fold_start-pd.Timedelta(days=1))
            bundle.update(schema=TRAINING_SCHEMA, source=source, fold=fold,
                          role="walk_forward_test", fit_warnings=messages,
                          frozen_params_sha256=frozen_hash, base_run=str(loaded["run_dir"]))
            save_verified_model(bundle, fold_dir/"models"/f"{name}.pkl",
                                predict_frame[FEATURES].iloc[:100])
            output = pd.DataFrame(dict(score=score, label=predict_frame["y_raw"],
                                      label_end_date=predict_frame["label_end_date"],
                                      model=name, role="walk_forward_test", fold=fold),
                                  index=predict_frame.index)
            # 允许标签晚于本预测窗口结尾，但不能越过研究终点。
            # 这只影响事后评价；不影响上面的训练与预测。
            usable = output.label.notna() & output.label_end_date.le(end)
            output["label_usable"] = usable
            output["evaluation_label"] = output["label"].where(usable)
            path = fold_dir/f"{name}_predictions.parquet"
            output.to_parquet(path)
            score_paths[name].append(path)
            row = dict(model=name, fold=fold, fit_seconds=seconds,
                       prediction_rows=len(output), labeled_rows=int(usable.sum()),
                       warnings=" | ".join(messages))
            if usable.any():
                metrics, daily = evaluate_model(output["evaluation_label"], score)
                row.update(metrics)
                daily.to_csv(fold_dir/f"{name}_daily.csv", index=False)
                print(f"  {name} 耗时={seconds:.1f}s RankIC={metrics['rank_ic_mean']:.5f}", flush=True)
            metrics_rows.append(row)
            pd.DataFrame(metrics_rows).to_csv(run_dir/"fold_metrics.csv", index=False)
            del estimator
        pd.DataFrame(window_rows).to_csv(run_dir/"windows.csv", index=False)
    pooled_rows, prediction_hashes = [], {}
    for name in names:
        pooled = pd.concat([pd.read_parquet(p) for p in score_paths[name]]).sort_index()
        if pooled.index.has_duplicates:
            raise ValueError("滚动预测窗口发生重复。")
        path = run_dir/f"{name}_predictions.parquet"
        pooled.to_parquet(path)
        prediction_hashes[name] = file_hash(path)
        row = dict(model=name, prediction_rows=len(pooled),
                   labeled_rows=int(pooled["label_usable"].sum()), role="walk_forward_test")
        if pooled["label_usable"].any():
            metrics, daily = evaluate_model(pooled["evaluation_label"], pooled["score"].to_numpy())
            row.update(metrics)
            daily.to_csv(run_dir/f"{name}_daily.csv", index=False)
        pooled_rows.append(row)
    if object_hash(frozen_params) != frozen_hash or source != training_source():
        raise ValueError("滚动过程中冻结参数或数据版本发生变化。")
    report = pd.DataFrame(pooled_rows)
    report.to_csv(run_dir/"model_comparison.csv", index=False)
    atomic_json(run_dir/"metadata.json", dict(status="complete", **config,
                prediction_sha256=prediction_hashes, folds=len(windows)))
    atomic_json(output_root/"latest_run.json", {"run_dir": run_dir.name})
    return run_dir, report


print("滚动训练接口已定义：run_rolling；未开始执行。")

In [ ]:
# 滚动训练会额外拟合六模型 × 滚动窗口数，按需开启。
RUN_ROLLING = False
ROLLING_TRAIN_MONTHS = None  # None=扩展窗口；24=最近24个月
ROLLING_RETRAIN_MONTHS = 3

if RUN_ROLLING:
    rolling_dir, rolling_report = run_rolling(
        base_run=loaded_models["run_dir"],
        start=SPLITS["test"][0],
        end=SPLITS["test"][1],
        retrain_months=ROLLING_RETRAIN_MONTHS,
        train_months=ROLLING_TRAIN_MONTHS,
        max_train_rows=TRAIN_MAX_ROWS,
    )
    display(rolling_report)
    print("滚动结果：", rolling_dir)
else:
    print("滚动训练未执行：RUN_ROLLING=False。")

## 9. 输出与下游接口

正式训练成功后更新 `data/processed/model_training/latest_run.json`，失败不会覆盖最新成功版本。每个 `run_...` 包含：

- `models/<模型>.pkl` 和同名 JSON：已拟合 Pipeline、预处理、参数、版本、可用时点与校验值。
- `best_params.json`、`config.json`、`metadata.json`：冻结参数、搜索预算、时间范围和来源版本。
- `cv_windows.csv`、`tuning_trials.csv`、`cv_fold_metrics.csv`：时间窗口、参数组和逐窗口指标。
- `model_comparison.csv`、`daily_metrics/`、`predictions/`：验证诊断和预测。
- `completed/`：已完成模型的检查点。配置相同才允许跳过已完成模型。

滚动结果位于 `model_training/rolling/run_.../`，包含 `fold-*/models/`、每折预测、`windows.csv`、`fold_metrics.csv`，以及每个模型合并后的 `<模型>_predictions.parquet`。索引为 `(datetime, instrument)`，包括 `score`、原始 `label`、`label_end_date`、`label_usable`、`evaluation_label`、`fold` 和 `role`。下游按日期和股票读取 `score`，评价使用 `evaluation_label`，避免把尚未实现的收益用于评价。

加载预测时默认禁止预测日期早于模型可用时点。`allow_historical=True` 只用于明确的历史诊断。新数据必须先生成同名且同序的原始因子，再交给保存的 Pipeline；不在预测端重新拟合填充或缩放。